# Imports

In [2]:
import re

import pandas as pd

pd.options.display.max_columns = 50

# Filenames

In [12]:
swissprot_fasta = "/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz"


# Sigfiles
swissprot_singleton_sig = f"{swissprot_fasta}.hp.k24.scale5.singleton.zip"
swissprot_aggregated_sig = f"{swissprot_fasta}.hp.k24.scale5.aggregated.zip"

manysketch_csv = f"{swissprot_fasta}.manysketch.csv"
print(manysketch_csv)

swissprot_aggregated_kmers_csv = f"{swissprot_aggregated_sig}.kmers.csv"

/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz.manysketch.csv


# Make files to input to manysketch

## Human

In [17]:
%%file /home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz.manysketch.csv
name,genome_filename,protein_filename
uniprot_sprot,,/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz

Overwriting /home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz.manysketch.csv


# Run Manysketch

In [10]:
! sourmash scripts manysketch  --help


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

usage:  manysketch [-h] [-q] [-d] -o OUTPUT [-p PARAM_STRING] [-c CORES] [-s]
                   [-f]
                   fromfile_csv

massively parallel sketching

positional arguments:
  fromfile_csv          a csv file containing paths to FASTA files. Columns
                        must be: 'name,genome_filename,protein_filename' or
                        'name,read1,read2'

options:
  -h, --help            show this help message and exit
  -q, --quiet           suppress non-error output
  -d, --debug           provide debugging output
  -o OUTPUT, --output OUTPUT
                        output zip file for the signatures
  -p PARAM_STRING, --param-string PARAM_STRING
                        parameter string for sketching (default:
                        k=31,scaled=1000)
  -c CORES, --cores CORES
                        number of cores to use (default is all available)
  -s, 

## Set parameters to share

In [4]:
parameters = "-p hp,k=24,scaled=5,abund"

### Make swissprot singleton sig (one sig per entry)

In [18]:
%%time

! sourmash scripts manysketch \
    $parameters \
    --singleton \
    -o $swissprot_singleton_sig \
    $manysketch_csv


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

=> sourmash_plugin_branchwater 0.9.14.dev0; cite Irber et al., doi: 10.1101/2022.11.02.514947

params: ['hp,k=24,scaled=5,abund']
sketching all files in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz.manysketch.csv' using 8 threads
Loaded 1 rows in total (0 genome and 1 protein files)
Building 1 sketch types:
    hp,k=24,scaled=5,num=0,abund=true
Starting file 1/1 (100%)
Writing manifest
DONE. Processed 1 fasta files
...manysketch is done! results in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz.hp.k24.scale5.singleton.zip'
CPU times: user 5.58 s, sys: 1.22 s, total: 6.8 s
Wall time: 13min 39s


### Make swissprot aggregated sig (one sig for all of swissprot)

In [21]:
%%time

! sourmash scripts manysketch \
    $parameters \
    -o $swissprot_aggregated_sig \
    $manysketch_csv


== This is sourmash version 4.8.14. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

=> sourmash_plugin_branchwater 0.9.14.dev0; cite Irber et al., doi: 10.1101/2022.11.02.514947

params: ['hp,k=24,scaled=5,abund']
sketching all files in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz.manysketch.csv' using 8 threads
Loaded 1 rows in total (0 genome and 1 protein files)
Building 1 sketch types:
    hp,k=24,scaled=5,num=0,abund=true
Starting file 1/1 (100%)
Writing manifest
DONE. Processed 1 fasta files
...manysketch is done! results in '/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz.hp.k24.scale5.aggregated.zip'
CPU times: user 1.64 s, sys: 321 ms, total: 1.96 s
Wall time: 3min 57s


# Get kmers for each hashval

In [22]:
! python sig2kmer.py --help

usage: sig2kmer.py [-h] [--output-sequences OUTPUT_SEQUENCES]
                   [--output-kmers OUTPUT_KMERS] [--input-is-protein] [-k K]
                   [--protein] [--no-protein] [--dayhoff] [--no-dayhoff]
                   [--hp] [--no-hp] [--skipm1n3] [--no-skipm1n3] [--skipm2n3]
                   [--no-skipm2n3] [--dna] [--no-dna]
                   query seqfiles [seqfiles ...]

positional arguments:
  query
  seqfiles

options:
  -h, --help            show this help message and exit
  --output-sequences OUTPUT_SEQUENCES
                        save matching sequences to this file.
  --output-kmers OUTPUT_KMERS
                        save matching kmers to this file.
  --input-is-protein    Consume protein sequences - no translation needed.
  -k K, --ksize K       k-mer size to select; no default.
  --protein             choose a protein signature; by default, a nucleotide
                        signature is used
  --no-protein          do not choose a protein signature
 

## Get UniProt kmers

In [23]:
%%time

! python sig2kmer.py \
    --output-kmers $swissprot_aggregated_kmers_csv \
    --no-dna \
    --input-is-protein \
    --hp \
    -k 24 \
    $swissprot_aggregated_sig \
    $swissprot_fasta

CPU times: user 10.3 s, sys: 2.37 s, total: 12.7 skmers in	572554 seqs from	/home/ec2-user/s3/seanome-kmerseek/applications/botryllus/data/uniprot/2025_01/uniprot_sprot.fasta.gz
Wall time: 21min 43s


### Read UniProt/SwissProt Kmer csv

In [25]:
import pandas as pd

swissprot_aggregated_kmers = pd.read_csv(swissprot_aggregated_kmers_csv)
print(swissprot_aggregated_kmers.shape)
swissprot_aggregated_kmers.head()

(38874668, 6)


,kmer_in_sequence,kmer_in_alphabet,hashval,start,read_name,filename
0,FSAEDVLKEYDRRRRMEALLLSLY,hphpphhpphppppphphhhhphh,3658408555502443395,2,sp|Q6GZX4|001R_FRG3G Putative transcription fa...,/home/ec2-user/s3/seanome-kmerseek/application...
1,EDVLKEYDRRRRMEALLLSLYYPN,pphhpphppppphphhhhphhhhp,2691840043550637581,5,sp|Q6GZX4|001R_FRG3G Putative transcription fa...,/home/ec2-user/s3/seanome-kmerseek/application...
2,DRRRRMEALLLSLYYPNDRKLLDY,ppppphphhhhphhhhpppphhph,1156837493005456783,12,sp|Q6GZX4|001R_FRG3G Putative transcription fa...,/home/ec2-user/s3/seanome-kmerseek/application...
3,MEALLLSLYYPNDRKLLDYKEWSP,hphhhhphhhhpppphhphpphph,1908013986383410926,17,sp|Q6GZX4|001R_FRG3G Putative transcription fa...,/home/ec2-user/s3/seanome-kmerseek/application...
4,LLSLYYPNDRKLLDYKEWSPPRVQ,hhphhhhpppphhphpphphhphp,271455203885221974,21,sp|Q6GZX4|001R_FRG3G Putative transcription fa...,/home/ec2-user/s3/seanome-kmerseek/application...


In [26]:
hp_kmer_counts = swissprot_aggregated_kmers.kmer_in_alphabet.value_counts()
hp_kmer_counts.head()

kmer_in_alphabet
pppppppppppppppppppppppp    48068
hppppppppppppppppppppppp     4387
pppppppppppppppppppppphp     3446
hhphhhhhhhhhhhhhhhhhhhhh     2722
hhhphhhhhhhhhhhhhhhhhhhh     2494
Name: count, dtype: int64

In [27]:
hashval_to_hp_kmer = pd.Series(
    dict(
        zip(
            swissprot_aggregated_kmers["kmer_in_alphabet"],
            swissprot_aggregated_kmers["hashval"],
        )
    )
)
# hashval_to_hp_kmer.index = hashval_to_hp_kmer.index.astype(int)
print(hashval_to_hp_kmer.shape)
hashval_to_hp_kmer.drop_duplicates()
print(hashval_to_hp_kmer.shape)

(3342529,)
(3342529,)


In [28]:
hp_kmer_to_hashval = pd.Series(
    dict(
        zip(
            swissprot_aggregated_kmers["hashval"],
            swissprot_aggregated_kmers["kmer_in_alphabet"],
        )
    )
)
# hashval_to_hp_kmer.index = hashval_to_hp_kmer.index.astype(int)
print(hp_kmer_to_hashval.shape)
hp_kmer_to_hashval.drop_duplicates()
print(hp_kmer_to_hashval.shape)

(3342529,)
(3342529,)


In [29]:
hashval_to_hp_kmer

hphpphhpphppppphphhhhphh    3658408555502443395
pphhpphppppphphhhhphhhhp    2691840043550637581
ppppphphhhhphhhhpppphhph    1156837493005456783
hphhhhphhhhpppphhphpphph    1908013986383410926
hhphhhhpppphhphpphphhphp     271455203885221974
                                   ...         
hhhpppphhhppppphpppphhhp    1607416551584701670
phhpppphhhppphphphhpphpp    3037327127502921112
pphphppphhhphhhhpphphhhp    3370396752700640464
hphhhphphhppppphhhpppppp    3123972378584791931
hpppphhphpphphhhphpppphh    2891155706973314033
Length: 3342529, dtype: int64

In [14]:
len("hphhphpppppppppphhpphhhh")

24

In [ ]:
hp_kmer_counts.query